# Notebook 9 Phase 4: Network CSV Construction for Gephi
**Epistolary Network Analysis — Luis Mitrovic Balbontín Archive**

### Data sources
- **NER data**: `Fase_3_Normalized_Propagated_Manual_Review_V3.csv`
  Post-normalization entity dataset (n=141 documents).
  PER entities extracted via Stanza 1.11.1 + gazetteer propagation
  + two-layer manual normalization.
- **Metadata**: `LuisMitrovic_CartasFase3.csv`
  Sender, recipient, date, language, and category per document.

### Network versions

This notebook produces two CSV files (nodes.csv, edges.csv) that feed directly
into Gephi. Three network versions were generated during the analysis:

- **V1** (weight ≥ 2, 222 edges): initial exploratory version, retained in
  `enterrenografoV1.gephi`
- **V2** (weight ≥ 1, partial normalizations): intermediate version
- **V3** (weight ≥ 1, 149 nodes, 669 edges): canonical version used in the
  thesis, with complete normalizations and corrected edge construction

The threshold change from V1 to V3 was motivated by the corpus size: in an
archive of 141 documents, single co-mentions carry independent evidential weight
as each document represents a distinct archival unit. V3 is the version
documented in the thesis and exported as `enterrenografoV3.gephi`.

### Network model
The network is **ego-centered**, with Luis Mitrovic Balbontín as
the central node. Following Imízcoz Beunza & Arroyo Ruiz (2011),
the network extends beyond direct correspondents to include persons
mentioned within the letters' content, revealing the broader social
universe documented in the archive.

Two layers of nodes are distinguished:
- **direct_correspondent**: appears as sender or recipient in metadata
- **mentioned_only**: appears only in NER PER_list

Edges represent **co-mention**: two persons appearing in the same
document. Edge weight equals the number of documents in which they
co-appear. As Ahnert et al. (2020, p. 76) note, every abstraction
involves a trade-off between information retained and analytical
tractability; this model prioritizes structural patterns over
directional letter flow. Node attributes (size, color) encode
meaningful variables following Venturini et al. (2019).

### Period classification
| Period | Years |
|---|---|
| european       | 1934–1939                  |
| chilean        | 1940–1971 / 1987–2006      |
| united_states  | 1972–1986                  |
| 2+ periods     | appears in 2+ periods      |
| unknown        | undated or outside periods |

**Inputs:**
- `data/raw/Fase_3_Normalized_Propagated_Manual_Review_V3.csv`
- `data/raw/LuisMitrovic_CartasFase3.csv`

**Outputs:**
- `data/network/nodes.csv`
- `data/network/edges.csv`

In [ ]:
from pathlib import Path

BASE       = Path.cwd().parent
DATA_RAW   = BASE / "data" / "raw"
DATA_PROC  = BASE / "data" / "processed"
DATA_NET   = BASE / "data" / "network"
FIGURES    = BASE / "figures"

for folder in [DATA_PROC, DATA_NET, FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"DATA_RAW:  {DATA_RAW}")
print(f"DATA_NET:  {DATA_NET}")

In [68]:
# Reproducibility 
import sys
print(f"Python  : {sys.version}")
print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print(f"\nOutput files:")
print(f"  {nodes_path}")
print(f"  {edges_path}")

Python  : 3.11.14 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 18:30:03) [MSC v.1929 64 bit (AMD64)]
pandas  : 3.0.0
numpy   : 2.4.2

Output files:
  C:\Users\felip\Tesis\Fase 4\nodes.csv
  C:\Users\felip\Tesis\Fase 4\edges.csv


In [62]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")

# Paths
NER_FILE  = DATA_RAW / "Fase_3_Normalized_Propagated_Manual_Review_V3.csv"
META_FILE = DATA_RAW / "LuisMitrovic_CartasFase3.csv"

# Load NER CSV
df_ner = pd.read_csv(NER_FILE, encoding="utf-8-sig", sep=",",
                     quotechar='"', quoting=0, engine="python")
df_ner = df_ner.dropna(how="all")
df_ner = df_ner[df_ner["Envelope"].notna()].reset_index(drop=True)

# Load metadata CSV
df_meta = pd.read_csv(META_FILE, encoding="utf-8-sig", sep=";",
                      quotechar='"', quoting=1, engine="python")
df_meta = df_meta[df_meta["Envelope"].notna()].reset_index(drop=True)

# Parse dates
df_meta["Date_parsed"] = pd.to_datetime(
    df_meta["Date"], format="%d-%m-%Y", errors="coerce"
)
df_meta["Year"] = df_meta["Date_parsed"].dt.year

# Parse PER entities
def parse_entities(cell):
    if pd.isna(cell) or str(cell).strip() == "":
        return []
    return [e.strip() for e in str(cell).split(";") if e.strip()]

df_ner["PER"] = df_ner["PER_list"].apply(parse_entities)

# Merge NER with metadata on Filename
df = df_ner.merge(
    df_meta[["Filename", "From", "To", "Date_parsed", "Year",
             "Language", "category", "include_in_network"]],
    on="Filename", how="left", suffixes=("", "_meta")
)
df = df.drop_duplicates(subset=["Filename", "Envelope"], keep="first")
df = df.reset_index(drop=True)

print(f" NER loaded     : {len(df_ner)} documents")
print(f" Metadata loaded: {len(df_meta)} documents")
print(f" Merged         : {len(df)} documents")
print(f"  Null dates     : {df['Year'].isna().sum()}")
print(f"\n── Language distribution ────────────────────────────")
print(df["Language"].value_counts().to_string())

✓ NER loaded     : 141 documents
✓ Metadata loaded: 141 documents
✓ Merged         : 141 documents
  Null dates     : 8

── Language distribution ────────────────────────────
Language
English    58
Spanish    46
German     37


In [63]:
# Reload metadata with corrections
df_meta = pd.read_csv(META_FILE, encoding="utf-8-sig", sep=";",
                      quotechar='"', quoting=1, engine="python")
df_meta = df_meta[df_meta["Envelope"].notna()].reset_index(drop=True)

df_meta["Date_parsed"] = pd.to_datetime(
    df_meta["Date"], format="%d-%m-%Y", errors="coerce"
)
df_meta["Year"] = df_meta["Date_parsed"].dt.year

# Re-merge with NER
df = df_ner.merge(
    df_meta[["Filename", "From", "To", "Date_parsed", "Year",
             "Language", "category", "include_in_network"]],
    on="Filename", how="left", suffixes=("", "_meta")
)
df = df.drop_duplicates(subset=["Filename", "Envelope"], keep="first")
df = df.reset_index(drop=True)

# Strip whitespace from include_in_network
df["include_in_network"] = df["include_in_network"].str.strip()
df["PER"] = df["PER_list"].apply(parse_entities)
df["ORG"] = df["ORG_list"].apply(parse_entities)

# Verify corrections
print(df[df["Envelope"] == "S88"]["include_in_network"].values)
print(f"── Total documents: {len(df)}")
print(f"\n── include_in_network distribution ─────────────────")
print(df["include_in_network"].value_counts().to_string())
print(f"\n── Unknown/None in From ─────────────────────────────")
unknowns = df[df["From"].isin(["Unknown", "None"]) | df["From"].isna()]
print(unknowns[["Envelope", "From", "To", "category",
                "include_in_network"]].to_string(index=False))

# Manual overrides for known exclusions 
# S88: non_epistolary_collected (yacht brochure) — excluded from network
# CSV parsing issue with quoted content prevents reading 'no' correctly
df.loc[df["Envelope"] == "S88", "include_in_network"] = "no"

# Verify
print(f"S88 include_in_network: {df[df['Envelope']=='S88']['include_in_network'].values[0]}")
print(f"\n── include_in_network final ─────────────────────────")
print(df["include_in_network"].value_counts().to_string())

<StringArray>
['yes']
Length: 1, dtype: str
── Total documents: 141

── include_in_network distribution ─────────────────
include_in_network
yes    140
no       1

── Unknown/None in From ─────────────────────────────
Envelope From  To                 category include_in_network
      S6  NaN NaN  non_epistolary_authored                yes
     S88  NaN NaN non_epistolary_collected                yes
   S122C  NaN NaN          personal_letter                yes
S88 include_in_network: no

── include_in_network final ─────────────────────────
include_in_network
yes    139
no       2


In [64]:
# nodes.csv (final clean version)

EGO = "Luis Mitrovic Balbontín"

EGO_VARIANTS = {
    "luis mitrovic balbontín", "luis mitrovic balbontin",
    "luis mitrovic", "mitrovic", "luis"
}

KNOWN_ORGS = {
    "Baburizza & Company LTD",
    "Club Alemán de Excursionismo",
    "Rectorate",
}

BABURIZZA_VARIANTS = {"Baburizza & Company Ltd.", "Baburizza & Co.", "Baburizza & Co"}
BABURIZZA_CANONICAL = "Baburizza & Company LTD"

CUCHO_VARIANTS = {"Cucho"}
CUCHO_CANONICAL = "Cucho Errázuriz"

EXTRA_NORMALIZATIONS = {
    "Jorge Hubner Gallo"     : "Jorge Hübner Gallo",
    "Jorge Hubner Bezanilla" : "Jorge Hübner Bezanilla",
    "John Dijkstra"          : "John G. Dijkstra",
    "Talao"                  : "Estanislao Echenique Guzmán",
    "Matuco"                 : "Matías Astoreca Sartori",
    "Matias Astoreca Sartori" : "Matías Astoreca Sartori",
}

PERIODS = {
    "european"      : [(1934, 1939)],
    "chilean"       : [(1940, 1971), (1987, 2006)],
    "united_states" : [(1972, 1986)],
}

def classify_period(years):
    years = {y for y in years if pd.notna(y)}
    if not years:
        return "unknown"
    active = set()
    for period, ranges in PERIODS.items():
        if any(start <= y <= end for y in years for start, end in ranges):
            active.add(period)
    return "unknown" if not active else ("2+ periods" if len(active) > 1 else active.pop())

HYPOTHESIS_ENTITIES = {
    "Etienne", "Bruno", "Gonzalo", "Franz Krandt", "Miss Italia"
}


def classify_primary_category(categories):
    if not categories:
        return "unknown"
    normalized = ["non_epistolary" if str(c).startswith("non_epistolary")
                  else c for c in categories]
    counts = Counter(normalized)
    top_cat, top_count = counts.most_common(1)[0]
    return top_cat if top_count / len(normalized) >= 0.6 else "mixed"

def get_attributes(doc_rows):
    years  = [r["Year"] for r in doc_rows if pd.notna(r.get("Year"))]
    langs  = [r["Language"] for r in doc_rows if pd.notna(r.get("Language"))]
    cats   = [r.get("category", "") for r in doc_rows
              if r.get("category") and pd.notna(r.get("category"))]
    period = classify_period(years)
    if langs:
        lc = Counter(langs)
        tl, tc = lc.most_common(1)[0]
        language = tl if tc / len(langs) >= 0.6 else "mixed"
    else:
        language = "unknown"
    primary_category = classify_primary_category(cats)
    return period, language, primary_category

# Build direct correspondents
direct_correspondents = set()
direct_corr_docs = defaultdict(list)

for _, row in df.iterrows():
    if row.get("include_in_network") == "no":
        continue
    for field in ["From", "To"]:
        val = row.get(field)
        if pd.notna(val):
            for name in str(val).split(";"):
                name = name.strip()
                if name and name != EGO and name.lower() not in EGO_VARIANTS:
                    if name in EXTRA_NORMALIZATIONS:
                        name = EXTRA_NORMALIZATIONS[name]
                    direct_correspondents.add(name)
                    direct_corr_docs[name].append(row)

# Manual additions: external document correspondents
# S122C: Gonzalo Díaz → Jorge Hübner Gallo
# Cannot be read from CSV due to internal line breaks in Content field
MANUAL_DIRECT_CORRESPONDENTS = {
    "Gonzalo Diaz": [{"Year": 1936.0, "Language": "Spanish",
                      "category": "personal_letter"}],
}

for name, doc_rows in MANUAL_DIRECT_CORRESPONDENTS.items():
    direct_correspondents.add(name)
    for row in doc_rows:
        direct_corr_docs[name].append(row)

# Build PER counter with normalizations
per_counter = Counter()
entity_docs = defaultdict(list)

for _, row in df.iterrows():
    for e in row["PER"]:
        e = e.strip()
        if e.lower() in EGO_VARIANTS:
            continue
        # Normalize BEFORE counting
        if e in CUCHO_VARIANTS:
            e = CUCHO_CANONICAL
        if e in EXTRA_NORMALIZATIONS:
            e = EXTRA_NORMALIZATIONS[e]
        # Count AFTER normalizing
        per_counter[e] += 1
        entity_docs[e].append(row)

# Build ORG counter with Baburizza normalization
org_counter = Counter()
df["ORG"] = df["ORG_list"].apply(parse_entities)
for _, row in df.iterrows():
    for e in row["ORG"]:
        e = e.strip()
        if e in BABURIZZA_VARIANTS:
            e = BABURIZZA_CANONICAL
        org_counter[e] += 1

# Build nodes
nodes = []

# Ego
nodes.append({
    "label"            : EGO,
    "type"             : "ego",
    "entity_type"      : "person",
    "primary_category" : "mixed",
    "mentions"         : 0,
    "period"           : "both",
    "language"         : "mixed",
    "is_identified"    : True,
})

all_entities = set(per_counter.keys()) | direct_correspondents

for ent in all_entities:
    if ent == EGO or ent.lower() in EGO_VARIANTS:
        continue

    is_direct     = ent in direct_correspondents
    node_type     = "direct_correspondent" if is_direct else "mentioned_only"
    entity_type   = "organization" if ent in KNOWN_ORGS else "person"
    is_identified = ent not in HYPOTHESIS_ENTITIES
    mentions      = per_counter.get(ent, 0)

    # Skip mentioned_only with < 2 mentions
    if node_type == "mentioned_only" and mentions < 2:
        continue

    all_docs = entity_docs.get(ent, []) + direct_corr_docs.get(ent, [])
    period, language, primary_category = get_attributes(all_docs)

    nodes.append({
        "label"            : ent,
        "type"             : node_type,
        "entity_type"      : entity_type,
        "primary_category" : primary_category,
        "mentions"         : mentions,
        "period"           : period,
        "language"         : language,
        "is_identified"    : is_identified,
    })

# Add KNOWN_ORGS not yet in nodes
existing_labels = {n["label"] for n in nodes}
for org in KNOWN_ORGS:
    if org in existing_labels:
        # Update entity_type if already added as person
        for n in nodes:
            if n["label"] == org:
                n["entity_type"] = "organization"
                n["mentions"] = org_counter.get(org, n["mentions"])
        continue
    org_docs = direct_corr_docs.get(org, [])
    period, language, primary_category = get_attributes(org_docs)
    nodes.append({
        "label"            : org,
        "type"             : "direct_correspondent",
        "entity_type"      : "organization",
        "primary_category" : primary_category,
        "mentions"         : org_counter.get(org, 0),
        "period"           : period,
        "language"         : language,
        "is_identified"    : True,
    })

df_nodes = pd.DataFrame(nodes).reset_index(drop=True)
df_nodes.insert(0, "id", range(1, len(df_nodes) + 1))

# Statistics 
print(f"── Total nodes: {len(df_nodes)}")
print(f"\n── By type ────────────")
print(df_nodes["type"].value_counts().to_string())
print(f"\n── By entity_type ─────────")
print(df_nodes["entity_type"].value_counts().to_string())
print(f"\n── By primary_category ───────")
print(df_nodes["primary_category"].value_counts().to_string())
print(f"\n── By period ───────")
print(df_nodes["period"].value_counts().to_string())
print(f"\n── Direct correspondents with 0 PER mentions ────────")
zero = df_nodes[(df_nodes["type"]=="direct_correspondent") &
                (df_nodes["mentions"]==0)]
print(f"  {len(zero)} nodes")
print(zero[["label","entity_type","period"]].to_string(index=False))

── Total nodes: 149

── By type ──────────────────────────────────────────
type
mentioned_only          89
direct_correspondent    59
ego                      1

── By entity_type ───────────────────────────────────
entity_type
person          146
organization      3

── By primary_category ──────────────────────────────
primary_category
personal_letter         123
institutional_letter     13
non_epistolary            5
mixed                     4
professional_letter       4

── By period ────────────────────────────────────────
period
european         77
chilean_early    21
chilean_late     21
both             17
unknown          13

── Direct correspondents with 0 PER mentions ────────
  23 nodes
                label  entity_type        period
         Pilola Celis       person       unknown
            Rectorate organization      european
           Unknown_S4       person       unknown
         Luis Arteaga       person      european
           Aunt Elisa       person      europea

In [65]:
# edges.csv construction

# Build node label → id lookup
label_to_id = {row["label"]: row["id"] for _, row in df_nodes.iterrows()}
node_labels  = set(label_to_id.keys())

# Determine node type lookup
label_to_type = {row["label"]: row["type"] for _, row in df_nodes.iterrows()}

def edge_type(src, tgt):
    """epistolary if both are ego or direct_correspondent, else enriched"""
    t1 = label_to_type.get(src, "mentioned_only")
    t2 = label_to_type.get(tgt, "mentioned_only")
    if t1 in {"ego", "direct_correspondent"} and \
       t2 in {"ego", "direct_correspondent"}:
        return "epistolary"
    return "enriched"

# Build co-mention edges
# Two entities co-appear in same document → edge
# Weight = number of documents where they co-appear

edge_counter  = Counter()   # (src_label, tgt_label) → doc count
edge_via_ego  = {}          # (src_label, tgt_label) → True/False

for _, row in df.iterrows():
    if row.get("include_in_network") == "no":
        continue

    # Get all PER entities in this doc that are in our node set
    # Apply same normalizations as in node construction
    def normalize_entity(e):
        e = e.strip()
        if e in CUCHO_VARIANTS:
            e = CUCHO_CANONICAL
        if e in EXTRA_NORMALIZATIONS:
            e = EXTRA_NORMALIZATIONS[e]
        return e

    doc_entities = [normalize_entity(e) for e in row["PER"]
                    if normalize_entity(e) in node_labels
                    and normalize_entity(e) != EGO]

    # Add ego if sender or recipient
    is_ego_doc = (
        str(row.get("From", "")) == EGO or
        str(row.get("To", "")) == EGO or
        any(v in str(row.get("From", "")) for v in ["Luis Mitrovic Balbontín"]) or
        any(v in str(row.get("To", "")) for v in ["Luis Mitrovic Balbontín"])
    )
    if is_ego_doc:
        doc_entities = [EGO] + doc_entities

    # Handle external docs (via_ego=True)
    from_val = str(row.get("From", ""))
    to_val   = str(row.get("To", ""))
    via_ego  = (EGO not in from_val and EGO not in to_val and
                row.get("From") is not None and row.get("To") is not None)

    # Add sender/recipient as entities if in node set
    for field_val in [from_val, to_val]:
        for name in field_val.split(";"):
            name = name.strip()
            if name in node_labels and name not in doc_entities:
                doc_entities.append(name)

    # Generate all pairs
    doc_entities = list(set(doc_entities))
    for a, b in combinations(sorted(doc_entities), 2):
        key = (a, b)
        edge_counter[key] += 1
        edge_via_ego[key]   = via_ego

# Build edges dataframe
edges = []
for (src, tgt), weight in edge_counter.items():
    if src not in label_to_id or tgt not in label_to_id:
        continue
    edges.append({
        "source"  : label_to_id[src],
        "target"  : label_to_id[tgt],
        "weight"  : weight,
        "type"    : edge_type(src, tgt),
        "via_ego" : edge_via_ego.get((src, tgt), False),
        "src_label": src,
        "tgt_label": tgt,
    })

df_edges_all = pd.DataFrame(edges)

# Apply threshold: weight >= 2 
df_edges = df_edges_all[df_edges_all["weight"] >= 1].reset_index(drop=True)

# Statistics
print(f"── Edges before threshold : {len(df_edges_all)}")
print(f"── Edges after threshold  : {len(df_edges)}")
print(f"\n── Breakdown by type ────────────────────────────────")
print(df_edges["type"].value_counts().to_string())
print(f"\n── via_ego edges ────────────────────────────────────")
print(df_edges["via_ego"].value_counts().to_string())
print(f"\n── Top 10 edges by weight ───────────────────────────")
print(df_edges.nlargest(10, "weight")
      [["src_label", "tgt_label", "weight", "type"]]
      .to_string(index=False))

── Edges before threshold : 669
── Edges after threshold  : 669

── Breakdown by type ────────────────────────────────
type
enriched      563
epistolary    106

── via_ego edges ────────────────────────────────────
via_ego
False    659
True      10

── Top 10 edges by weight ───────────────────────────
                src_label               tgt_label  weight       type
  Luis Mitrovic Balbontín      Winold Tjark Reiss      35 epistolary
              Angela Gana Luis Mitrovic Balbontín      27 epistolary
  Luis Mitrovic Balbontín        Rolf Altenburger      20 epistolary
  Luis Mitrovic Balbontín               Nora Siré      13 epistolary
Gonzalo Latorre Salamanca Luis Mitrovic Balbontín      10 epistolary
                Nora Siré      Winold Tjark Reiss       8 epistolary
              Angela Gana      Winold Tjark Reiss       8 epistolary
              Angela Gana Consuelo Gana Snodgrass       8   enriched
  Consuelo Gana Snodgrass Luis Mitrovic Balbontín       8   enriched
      

In [66]:
# Export nodes.csv and edges.csv

# Final nodes, drop helper columns
df_nodes_export = df_nodes[[
    "id", "label", "type", "entity_type", "primary_category",
    "mentions", "period", "language", "is_identified"
]].copy()

# Final edges — drop label columns (Gephi uses ids)
df_edges_export = df_edges[[
    "source", "target", "weight", "type", "via_ego"
]].copy()

# Export
nodes_path = DATA_NET / "nodes.csv"
edges_path = DATA_NET / "edges.csv"

df_nodes_export.to_csv(nodes_path, index=False, encoding="utf-8-sig")
df_edges_export.to_csv(edges_path, index=False, encoding="utf-8-sig")

print(f"✓ Exported: {nodes_path}")
print(f"  {len(df_nodes_export)} nodes")
print(f"\n✓ Exported: {edges_path}")
print(f"  {len(df_edges_export)} edges")

# Preview 
print(f"\n── nodes.csv sample ──────")
print(df_nodes_export.head(10).to_string(index=False))
print(f"\n── edges.csv sample ──────")
print(df_edges_export.head(10).to_string(index=False))

✓ Exported: C:\Users\felip\Tesis\Fase 4\nodes.csv
  149 nodes

✓ Exported: C:\Users\felip\Tesis\Fase 4\edges.csv
  669 edges

── nodes.csv sample ─────────────────────────────────
 id                   label                 type  entity_type     primary_category  mentions        period language  is_identified
  1 Luis Mitrovic Balbontín                  ego       person                mixed         0          both    mixed           True
  2                    Phil       mentioned_only       person      personal_letter         3 chilean_early  Spanish           True
  3             Cota Vicuña       mentioned_only       person      personal_letter         2      european  English           True
  4              Spengemann       mentioned_only       person      personal_letter         3      european   German           True
  5    Miguel Dávila Carson direct_correspondent       person  professional_letter         2      european   German           True
  6            Pilola Celis direct

## Network V1 summary

### Nodes (n=147)

| Type | Count |
|---|---|
| ego | 1 |
| direct_correspondent | 58 |
| mentioned_only | 88 |
| **Total** | **147** |

| Period | Count |
|---|---|
| european | — |
| chilean | — |
| united_states | — |
| 2+ periods | — |
| unknown | — |

### Edges (n=662)

| Type | Count |
|---|---|
| epistolary | — |
| enriched | — |
| **Total** | **662** |

Threshold applied: weight ≥ 2 (co-appearance in 2+ documents).
460 edges below threshold excluded.

### Network density (approximate)
Density = 2E / N(N-1) = 2×222 / 150×149 = **0.020**

A density of ~2% is typical for ego-centered historical
correspondence networks of this scale, consistent with
Imízcoz Beunza & Arroyo Ruiz (2011).

### Top 10 nodes by degree (to be computed in Gephi)
Degree centrality will be computed in Gephi after import.
Expected high-degree nodes based on edge weights:
Luis Mitrovic Balbontín (ego), Winold Tjark Reiss,
Angela Gana, Rolf Altenburger, Nora Siré.

In [67]:
# Top 10 nodes by degree 
from collections import defaultdict

degree = defaultdict(int)
for _, row in df_edges.iterrows():
    degree[row["source"]] += 1
    degree[row["target"]] += 1

degree_df = pd.DataFrame([
    {"id": k, "degree": v} for k, v in degree.items()
]).merge(df_nodes[["id", "label", "type"]], on="id")

print("── Top 10 nodes by degree ────")
print(degree_df.nlargest(10, "degree")
      [["label", "type", "degree"]]
      .to_string(index=False))

print("\n── Top 10 edges by weight ────")
print(df_edges.nlargest(10, "weight")
      [["src_label", "tgt_label", "weight", "type"]]
      .to_string(index=False))

print(f"\n── Network statistics ───")
n = len(df_nodes)
e = len(df_edges)
density = (2 * e) / (n * (n - 1))
print(f"  Nodes   : {n}")
print(f"  Edges   : {e}")
print(f"  Density : {density:.4f}")

── Top 10 nodes by degree ───────────────────────────
                            label                 type  degree
          Luis Mitrovic Balbontín                  ego     143
                      Angela Gana direct_correspondent      52
               Winold Tjark Reiss direct_correspondent      37
          Consuelo Gana Snodgrass       mentioned_only      32
                 Rolf Altenburger direct_correspondent      30
Cornelius Stribling Snodgrass Sr.       mentioned_only      26
                    Ford Bartlett       mentioned_only      26
      Estanislao Echenique Guzmán       mentioned_only      24
                    Peter Behrens direct_correspondent      23
                        Raúl Vial       mentioned_only      23

── Top 10 edges by weight ───────────────────────────
                src_label               tgt_label  weight       type
  Luis Mitrovic Balbontín      Winold Tjark Reiss      35 epistolary
              Angela Gana Luis Mitrovic Balbontín      27 ep

## Methodological notes of Network V1: CSV Construction

### Data sources
- **NER data**: `Fase_3_Normalized_Propagated_Manual_Review_V3.csv`
  Post-normalization entity dataset (n=141 documents, 140 active in network).
  PER entities extracted via Stanza 1.11.1 + gazetteer propagation from
  Fase 2 + two-layer manual normalization.
- **Metadata**: `LuisMitrovic_CartasFase3.csv`
  Sender, recipient, date, language, and category per document.
  Join performed on `Filename` (not `Envelope`) to preserve document-level
  granularity, since one envelope may contain multiple letters of different dates.

### Node construction decisions

#### Type classification
- **ego**: Luis Mitrovic Balbontín — single central node.
  Variants normalized: "Luis Mitrovic", "Luis", "Mitrovic" → ego.
- **direct_correspondent**: appears as sender or recipient in metadata.
  Always included regardless of mention count threshold.
  53 direct correspondents identified from `From`/`To` fields.
- **mentioned_only**: appears only in NER PER_list.
  Subject to 2+ mention threshold.

#### Entity type
- `person`: default for all PER entities
- `organization`: manually assigned to correspondents identified as
  institutions (`Baburizza & Company LTD`, `Club Alemán de Excursionismo`,
  `Rectorate`). Organizations included because they document real
  relationships: financial (Baburizza), social/athletic (Club Alemán),
  institutional (Rectorate).
- Organization mentions counted from ORG_list, not PER_list.
  Baburizza variants normalized: "Baburizza & Co.", "Baburizza & Co",
  "Baburizza & Company Ltd." → "Baburizza & Company LTD" (5 total mentions).

#### Primary category
Derived from the predominant document category of letters where the
entity appears (≥60% threshold → category label; otherwise "mixed").
Categories: personal_letter, institutional_letter, professional_letter,
non_epistolary, mixed. Encodes the nature of the documented relationship,
not the entity type itself.

#### Period classification
Based on years of documents where the entity appears:
- `european`: 1934–1938
- `chilean`: 1939–1971 / 1986-2006
- `united_states`: 1972–1986
- `both`: appears in 2+ defined periods
- `unknown`: undated documents, or years outside defined periods

Note: period reflects documented interaction, not biographical origin
of the correspondent. A Chilean correspondent mentioned in European-period
letters is classified as `european`.

#### Normalizations applied
- `"Cucho"` → `"Cucho Errázuriz"` (direct correspondent normalization)
- `"Luis Mitrovic"`, `"Luis"`, `"Mitrovic"` → ego node (collapsed)
- `"Luis Mitrovic Puljizević"` included as `direct_correspondent`
  (father of ego; documentary presence in archive)

#### Excluded nodes
- `"Unknown"` (generic unidentified node) — excluded: no verifiable identity
- `Unknown_S88` — excluded: S88 is a yacht brochure (`non_epistolary_collected`,
  `include_in_network=no`). CSV parsing issue with internal line breaks
  required manual override in code.

#### Unverified/hypothesis entities (`is_identified=False`)
Etienne, Bruno, Gonzalo, Franz Krandt, Miss Italia — retained with
`is_identified=False` as documented in thesis §3.5.

### Edge construction decisions

#### Co-mention model
An edge exists between two entities if they appear in the same document.
Weight = number of documents in which they co-appear.
Rationale: co-mention in a letter implies shared social context —
the writer knew both parties well enough to reference them together.
Following Imízcoz Beunza & Arroyo Ruiz (2011), the network extends
beyond direct correspondents to include mentioned persons, revealing
the broader social universe documented in the archive.

#### Edge type
- `epistolary`: both nodes are ego or direct_correspondent —
  documents a verified letter exchange relationship
- `enriched`: at least one node is mentioned_only —
  documents a contextual co-presence inferred from NER

#### External documents (via_ego)
Three documents where Luis Mitrovic is neither sender nor recipient:
- S10: O.B.Y. → Luis Mitrovic Puljizević (father)
- S89: Luis Mitrovic Puljizević → Alexander Popp
- S90A: Miguel Dávila Carson → Emil Hoppe; Otto Schönthal

These generate direct edges between their actors (`via_ego=True` attribute).
Their presence in the archive implies Luis Mitrovic's role as custodian
or intermediary — itself a network signal.

### Threshold decisions

#### Node threshold for V1: ≥2 mentions (direct_correspondent always included)

A minimum of 2 mentions was applied to `mentioned_only` entities to reduce
noise from single-occurrence extractions that may represent transcription
errors, OCR artifacts, or incidental references with no structural significance
in the network. 

Direct correspondents are exempt from this threshold by design: their
presence in the `From`/`To` fields of the metadata constitutes independent
documentary evidence of a relationship with the ego node, regardless of
how frequently they are mentioned in the letter content. A correspondent
who exchanged a single letter with Mitrovic is structurally different from
a name that appears once in passing — the former documents an act of
communication, the latter may simply document a cultural reference.

Of the 449 unique PER entities identified before filtering, 334 had fewer
than 2 mentions and were excluded. The retained 115 mentioned_only nodes,
combined with 55 direct_correspondents and the ego node, yield a network
of 150 nodes that balances coverage with analytical tractability.

#### Edge threshold: weight ≥2 (co-appear in 2+ documents)

An edge weight of 1 — two entities appearing together in a single document
— was excluded as insufficient evidence of a structural connection. A
single co-mention may reflect a coincidental reference rather than a
meaningful social proximity. Weight ≥2 requires that the co-appearance
be repeated across at least two independent documents, providing a stronger
signal of sustained contextual association.

Of the 682 candidate edges generated before filtering, 460 had weight=1
and were excluded. The retained 222 edges represent co-appearances
confirmed across multiple documents, forming a more reliable basis
for structural analysis.

Both thresholds are conservative choices that prioritize network
interpretability over completeness. They are documented explicitly
as methodological decisions rather than technical necessities, following
the principle established in Ahnert et al. (2020) that every parameter
choice in network construction carries historiographical implications.

### Network statistics
| Metric | Value |
|---|---|
| Nodes | 150 |
| Edges | 222 |
| Density | 0.0199 (~2%) |
| Ego degree | 82 |
| Max edge weight | 35 (Luis Mitrovic ↔ Winold Tjark Reiss) |

Density of ~2% is consistent with ego-centered historical correspondence
networks of comparable scale (Imízcoz Beunza & Arroyo Ruiz, 2011).

### Trade-offs acknowledged

#### Structural patterns over directional letter flow
The network is modeled as **undirected and weighted**, treating a letter
from A to B as equivalent in structure to a letter from B to A. This
sacrifices directionality — who initiates correspondence, who responds,
who is silent — in exchange for computational tractability and
comparability across the full corpus. Directed network analysis would
require complete metadata for all 141 documents, which is not uniformly
available given the archival gaps and partially identified correspondents
documented in §3.5. The undirected model is appropriate for the present
analysis, which focuses on the identification of structural clusters and
prominent intermediaries rather than the dynamics of epistolary initiative.

#### Co-mention as proxy for social proximity
The co-mention model assumes that two persons appearing in the same letter
share a social context known to the writer. This is a strong but defensible
assumption in the context of personal correspondence: a writer who mentions
two people in the same letter typically knows them through overlapping
social circles, shared experiences, or mutual relationships. This follows
the epistolary network methodology of Imízcoz Beunza & Arroyo Ruiz (2011),
who demonstrate that mentioned persons — not only direct correspondents —
are essential to reconstructing the full social universe documented in a
personal archive.

The limitation of this proxy is that co-mention does not distinguish
between a close shared relationship and a mere contextual association.
Two people mentioned in the same letter about a ski trip in 1936 may
have known each other directly, or may simply have both been present
at the same event without personal connection. This ambiguity is
inherent to the co-mention model and cannot be resolved without
additional documentary evidence.

#### Corpus coverage over precision
The NER pipeline, even after two-layer normalization, retains residual
noise: unresolved name variants, partially identified entities
(is_identified=False), and potential false positives that survived
the normalization process. The decision to retain these entities rather
than exclude them conservatively reflects a commitment to corpus coverage:
in a small archive of 141 documents, over-filtering risks erasing
meaningful signal along with noise. The `is_identified` attribute
in nodes.csv allows downstream analysis to filter hypothesis entities
when precision is required, while preserving them for exploratory
visualization in Gephi.

### Output files
| File | Contents |
|---|---|
| `nodes.csv` | 150 nodes with 9 attributes |
| `edges.csv` | 222 edges with 5 attributes |

## Modularity Class Analysis: Network V2 (weight ≥ 1)
> **Note:** This analysis corresponds to an intermediate network version with
> partial normalizations. The canonical V3 network (enterrenografoV3.gephi)
> produced modularity = 0.33 with 11 communities. The community interpretations
> below are retained as process documentation and updated in Chapter 4 of the thesis.
 
Modularity score: **0.373** (resolution 0.5, weighted)
Communities detected: **14** (5 major + 9 minor)

### Major communities (>5% of nodes)

| Class | Nodes | % | Key figures | Interpretation |
|---|---|---|---|---|
| 0 | 41 | 27.3% | Winold Tjark Reiss, Nora Siré, Etienne, Mary | **Anglo-Chilean circle** — long-term correspondents across multiple periods |
| 8 | 24 | 16.0% | Peter Behrens, Daisy zu Dohna, Alexander Popp, Luis Mitrovic Puljizević | **Bauhaus/Vienna circle** — formative European period, masters and peers |
| 9 | 22 | 14.7% | Consuelo Gana Snodgrass, Cornelius Stribling Snodgrass Sr., Ford Bartlett, Cucho Errázuriz | **Chilean-American circle** — Snodgrass family + Chilean elite |
| 6 | 16 | 10.7% | Gonzalo Latorre, Baburizza & Company, Georgia Balbontín, Franica Abovic | **Family/Chilean-European circle** — family connections and Chilean contacts in Europe |
| 12 | 12 | 8.0% | Rolf Altenburger, Dijkstra family | **Late German circle** — post-return correspondence with German contacts |
| 7 | 9 | 6.0% | Angela Gana, Roland Palmedo | **Inner circle** — Angela Gana as highest betweenness node after ego |

### Key findings

#### Betweenness centrality — top intermediaries
| Node | Betweenness | Type | Community |
|---|---|---|---|
| Luis Mitrovic Balbontín | 8,577 | ego | — |
| Angela Gana | 421 | direct_correspondent | 7 |
| Winold Tjark Reiss | 232 | direct_correspondent | 0 |
| Peter Behrens | 126 | direct_correspondent | 8 |
| Luis Mitrovic Puljizević | 62 | direct_correspondent | 8 |
| Consuelo Gana Snodgrass | 62 | mentioned_only | 9 |
| Rolf Altenburger | 139 | direct_correspondent | 12 |

#### Notable observations
- **Angela Gana** has the highest betweenness after the ego (421) —
  she is the most important structural intermediary in the network,
  bridging the inner circle with multiple other communities.
- **Luis Mitrovic Puljizević** (father) clusters with the Bauhaus/Vienna
  circle (Class 8), consistent with his role in connecting Luis to
  the European academic world.
- **Class 3** (Eleanor Lee Potts, Cornelius Stribling Snodgrass Jr., Susan)
  forms an isolated cluster — the Snodgrass Jr. branch appears structurally
  separate from the Sr. branch (Class 9), suggesting different relational
  contexts within the same family network.
- **Class 1** contains duplicate normalization artifact
  (Miguel Dávila Carson / Miguel Davila Carson) — to be resolved
  in next normalization cycle.

### Minor communities (≤5% of nodes)
Classes 1–5, 10–13 contain 1–4 nodes each, representing peripheral
figures or isolated correspondence relationships. Class 10
(Ana María Migliassi, Fernando Migliassi) documents a family
correspondence cluster with no connections to other communities
except through the ego node.

In [69]:
# All external documents (ego not sender or recipient)
external = df[
    (~df["From"].fillna("").str.contains("Luis Mitrovic Balbontín")) &
    (~df["To"].fillna("").str.contains("Luis Mitrovic Balbontín")) &
    (df["include_in_network"] == "yes") &
    (df["From"].notna()) &
    (df["To"].notna())
]
print(f"External docs: {len(external)}")
print(external[["Envelope", "From", "To", "category"]].to_string(index=False))

External docs: 3
Envelope                     From                        To             category
     S10                 O. B. Y.  Luis Mitrovic Puljizević institutional_letter
     S89 Luis Mitrovic Puljizević            Alexander Popp institutional_letter
    S90A     Miguel Dávila Carson Emil Hoppe;Otto Schönthal  professional_letter


In [70]:
# Verify normalizations
checks = [
    "Matías Astoreca Sartori", "Matuco",
    "Estanislao Echenique Guzmán", "Talao",
    "Jorge Hübner Gallo", "Jorge Hubner Gallo",
    "John G. Dijkstra", "John Dijkstra",
    "Gonzalo Diaz",
]
for name in checks:
    match = df_nodes[df_nodes["label"] == name]
    if len(match) > 0:
        print(f"✓ '{name}': id={match['id'].values[0]}, mentions={match['mentions'].values[0]}, type={match['type'].values[0]}")
    else:
        print(f"✗ '{name}': NOT FOUND")

✓ 'Matías Astoreca Sartori': id=68, mentions=4, type=direct_correspondent
✗ 'Matuco': NOT FOUND
✓ 'Estanislao Echenique Guzmán': id=107, mentions=5, type=mentioned_only
✗ 'Talao': NOT FOUND
✓ 'Jorge Hübner Gallo': id=35, mentions=4, type=direct_correspondent
✗ 'Jorge Hubner Gallo': NOT FOUND
✓ 'John G. Dijkstra': id=31, mentions=5, type=direct_correspondent
✗ 'John Dijkstra': NOT FOUND
✓ 'Gonzalo Diaz': id=121, mentions=1, type=direct_correspondent


In [71]:
# Check edges for normalized nodes
for name in ["Estanislao Echenique Guzmán", "Matías Astoreca Sartori"]:
    edges = df_edges[(df_edges["src_label"] == name) | 
                     (df_edges["tgt_label"] == name)]
    print(f"{name}: {len(edges)} edges")
    print(edges[["src_label", "tgt_label", "weight"]].to_string(index=False))
    print()

Estanislao Echenique Guzmán: 24 edges
                        src_label                   tgt_label  weight
                      Angela Gana Estanislao Echenique Guzmán       2
                       Anita Lyon Estanislao Echenique Guzmán       1
                           Blaise Estanislao Echenique Guzmán       1
          Consuelo Gana Snodgrass Estanislao Echenique Guzmán       2
                    Ernesto Pinto Estanislao Echenique Guzmán       1
      Estanislao Echenique Guzmán                      Felipe       2
      Estanislao Echenique Guzmán               Ford Bartlett       2
      Estanislao Echenique Guzmán                      Louise       1
      Estanislao Echenique Guzmán     Luis Mitrovic Balbontín       3
      Estanislao Echenique Guzmán                   Raúl Vial       2
                 Betty Schönmeyer Estanislao Echenique Guzmán       1
Cornelius Stribling Snodgrass Sr. Estanislao Echenique Guzmán       1
      Estanislao Echenique Guzmán                   

In [72]:
# Check entity_docs for normalized names
for name in ["Estanislao Echenique Guzmán", "Matías Astoreca Sartori",
             "Talao", "Matuco"]:
    count = len(entity_docs.get(name, []))
    print(f"'{name}' in entity_docs: {count} documents")

# Check per_counter
for name in ["Estanislao Echenique Guzmán", "Matías Astoreca Sartori",
             "Talao", "Matuco"]:
    print(f"'{name}' in per_counter: {per_counter.get(name, 0)}")

'Estanislao Echenique Guzmán' in entity_docs: 5 documents
'Matías Astoreca Sartori' in entity_docs: 4 documents
'Talao' in entity_docs: 0 documents
'Matuco' in entity_docs: 0 documents
'Estanislao Echenique Guzmán' in per_counter: 5
'Matías Astoreca Sartori' in per_counter: 4
'Talao' in per_counter: 0
'Matuco' in per_counter: 0


In [1]:

ner  = pd.read_csv(DATA_RAW / "Fase_3_Normalized_Propagated_Manual_Review_V3.csv",
                   encoding="utf-8-sig")
meta = pd.read_csv(DATA_RAW / "LuisMitrovic_CartasFase3.csv",
                   sep=";", encoding="utf-8-sig")

all_per = set()
for v in ner['PER_list'].dropna():
    for p in str(v).split(';'):
        p = p.strip()
        if p: all_per.add(p)

from_to = set()
for v in meta['From'].dropna():
    for p in str(v).split(';'):
        p = p.strip()
        if p and p not in ('Unknown','None',''): from_to.add(p)
for v in meta['To'].dropna():
    for p in str(v).split(';'):
        p = p.strip()
        if p and p not in ('Unknown','None',''): from_to.add(p)

print(f"Unique PER in NER: {len(all_per)}")
print(f"Unique From/To in metadata: {len(from_to)}")
print(f"PER not in From/To: {len(all_per - from_to)}")

FileNotFoundError: [Errno 2] No such file or directory: 'Fase_3_Normalized_Propagated_Manual_Review_V3.csv'

## Network summary (V3 — canonical)

### Nodes (n=149)

| Type | Count |
|---|---|
| ego | 1 |
| direct_correspondent | 58 |
| mentioned_only | 90 |
| **Total** | **149** |

| Entity type | Count |
|---|---|
| person | 146 |
| organization | 3 |

| Period | Count |
|---|---|
| european | — |
| chilean | — |
| united_states | — |
| 2+ periods | — |
| unknown | — |

### Edges (n=669)

| Type | Count |
|---|---|
| epistolary | 93 |
| enriched | 576 |
| **Total** | **669** |

Threshold applied: weight ≥ 1.

### Network density
Density = 2E / N(N-1) = 2×669 / 149×148 = **0.061**

Density of ~6% is consistent with ego-centered historical
correspondence networks of comparable scale.